In [1]:
from dotenv import load_dotenv
from gql import Client, gql
from gql.transport.requests import RequestsHTTPTransport
import numpy as np
import os
import pandas as pd

In [2]:
load_dotenv()
ACCESS_TOKEN = os.environ["YELP_API_TOKEN"]
HEADERS = {"Authorization": "Bearer " + ACCESS_TOKEN}
TRANSPORT = RequestsHTTPTransport(
    url="https://api.yelp.com/v3/graphql", headers=HEADERS)
CLIENT = Client(transport=TRANSPORT, fetch_schema_from_transport=False)

In [3]:
def get_gyms(categories, location, radius):
    query = gql(
        """
      {
        search(categories: "%s",
              location: "%s"
              radius: %d) {
          total
        }
      }
  """
        % (categories, location, np.minimum(radius, 40_000)))

    return CLIENT.execute(query)

In [4]:
df = pd.read_csv("../cities.csv", index_col=[0, 1])

data = pd.DataFrame(index=df.index)
data["population"] = df["2023 estimate"]
data["area"] = pd.to_numeric(df["land area"], errors="coerce")

# convert from sq mi to sq m, a = pi*r^2
data["radius"] = np.sqrt(data["area"]*2589988.1103/np.pi)

data.head()

,,population,area,radius
city,state,,,
New York,New York,8258035,300.5,15739.690454
Los Angeles,California,3820914,469.5,19673.958328
Chicago,Illinois,2664452,227.7,13701.100885
Houston,Texas,2314157,640.4,22977.332585
Phoenix,Arizona,1650070,518.0,20665.162692


In [5]:
data["gyms"] = 0

for i, row in data.iterrows():
    location = "%s, %s" % (i[0], i[1])
    # print(location)
    try:
        result = get_gyms("rock_climbing", location, row["radius"])
        data.loc[i, "gyms"] = int(result["search"]["total"])
    except Exception as e:
        print(e)

cannot convert float NaN to integer


In [6]:
climbing = pd.read_csv("climbing.csv", index_col=[0, 1])

data["per_100k"] = data["gyms"] / data["population"] * 100_000

chicago = data.loc[("Chicago", "Illinois"), "per_100k"]
data["score"] = np.log2(data["per_100k"] / chicago)

data.sort_values("score", ascending=False, inplace=True)

/home/dan/danwahl/schelling-out/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [7]:
print(data.to_string())
data.to_csv("climbing.csv")

                                       population   area        radius  gyms   per_100k     score
city             state                                                                           
Boulder          Colorado                  105898   26.3   4656.416539    14  13.220269  5.138518
Berkeley         California                118962   10.4   2928.131782     8   6.724837  4.163338
Westminster      Colorado                  114875   31.6   5104.080204     7   6.093580  4.021129
Inglewood        California                102865    9.1   2739.016478     6   5.832888  3.958049
Burbank          California                102755   17.3   3776.565317     5   4.865943  3.696558
St. George       Utah                      104578   78.5   8044.680069     5   4.781120  3.671188
Centennial       Colorado                  106883   29.7   4948.256155     5   4.678012  3.639735
Costa Mesa       California                108354   15.8   3609.129724     5   4.614504  3.620015
Lakewood         Col

In [8]:
pd.to_numeric(data["population"], errors="coerce")

city            state     
Boulder         Colorado      105898
Berkeley        California    118962
Westminster     Colorado      114875
Inglewood       California    102865
Burbank         California    102755
                               ...  
Corpus Christi  Texas         316595
Edinburg        Texas         105799
Sandy Springs   Georgia       105793
Renton          Washington    104491
Lee's Summit    Missouri      104184
Name: population, Length: 336, dtype: int64